# TBP22 22CI Hybrid Asm Dataset - Gubbins Results Processing - Step 1

# Import statements

In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns
#import pickle

%matplotlib inline

In [2]:
# https://bioframe.readthedocs.io/en/latest/guide-intervalops.html
import bioframe as bf


In [3]:
import json

import ete3 as ETE

from ete3 import Tree


### Import custom functions

In [4]:
#from gcutils.general import parse_PAFtools_VarTSV, label_DF_ByOvrLapGenes, label_PAF_DF_ByOvrLapGenes'

from gcutils.gubbinsfuncs import get_RecombEvents_From_Gubbins_GFF

from gcutils.gubbinsfuncs import parse_BaseReconstruction_EMBL_Gubbins, annotate_Gubbins_SNP_Events_By_RecombEventID

from gcutils.gubbinsfuncs import get_RecombEvents_From_Gubbins_GFF_V2, label_Gubbins_Events_DF_ByOvrLap_H37RvGenes, label_Gubbins_Events_DF_ByOvrLap_HHRs

from gcutils.gubbinsfuncs import annotate_HHR_with_GCE_overlaps

from gcutils.gubbinsfuncs import get_BranchLengths_and_DescendantCounts_FromTree



from gcutils.general import check_overlap_with_gene_group 

In [5]:
RE_CoordCols = ("seqname", "start_0based", "end_1based")
HmRegion_CoordCols = ("Chr", "Start", "End")


#### Set matplotlib text export settings for Adobe Illustrator

In [6]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

#### Pandas Viewing Settings

In [7]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

# Import/parse processed H37rv genome annotations

In [8]:
RepoRef_Dir = "../../References"

AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir = f"{RepoRef_Dir}/201027_H37rv_AnnotatedGenes_And_IntergenicRegions"
H37Rv_GenomeAnnotations_Genes_TSV = f"{AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir}/H37Rv_GenomeAnnotations.Genes.tsv"

## H37Rv Gene Annotations TSV
H37Rv_GenomeAnno_Genes_DF = pd.read_csv(H37Rv_GenomeAnnotations_Genes_TSV, sep = "\t")
H37Rv_GeneInfo_Subset_DF = H37Rv_GenomeAnno_Genes_DF[["H37rv_GeneID", "Symbol", "Feature", "Functional_Category", "Is_Pseudogene", "Product", "PEandPPE_Subfamily", "ExcludedGroup_Category"]]

RvID_To_Symbol_Dict = dict(H37Rv_GeneInfo_Subset_DF[['H37rv_GeneID', 'Symbol']].values)

ESX_Genes_List_TSV = f"{RepoRef_Dir}/190927_H37rv_ListOf_ESXgenes.tsv"
Esx_Genes_DF = pd.read_csv(ESX_Genes_List_TSV, sep = '\t')

In [9]:
H37Rv_GenomeAnno_Genes_DF.head(1)

,Chrom,Start,End,Strand,H37rv_GeneID,Symbol,Feature,Functional_Category,Is_Pseudogene,Product,PEandPPE_Subfamily,ExcludedGroup_Category
0,NC_000962.3,0,1524,+,Rv0001,dnaA,CDS,information pathways,No,Chromosomal replication initiator protein DnaA,NaN,NotExcluded


### Define paths to H37Rv genome masking schemes

In [10]:
RepoRef_Dir = "../../References"

MaskingSchemes_Dir = f"{RepoRef_Dir}/Mtb_H37Rv_MaskingSchemes"
PLC_Scheme_BED = f"{MaskingSchemes_Dir}/201027_Mtb_H37rv_pLC_Regions_CoscollaExcludedGenes.bed"


## Define relevant H37Rv gene lists for analysis (PE/PPE, Esx, 13E12 gene)

In [11]:
ListOf_Esx_Symbols = list(Esx_Genes_DF["symbol"].values)
ListOf_Esx_RvIDs = list(Esx_Genes_DF["gene_id"].values)

In [12]:
listOf_PEPPE_Symbols = list( H37Rv_GenomeAnno_Genes_DF.query(" Functional_Category == 'PE/PPE' ")["Symbol"].values )
listOf_PEPPE_RvIDs = list( H37Rv_GenomeAnno_Genes_DF.query(" Functional_Category == 'PE/PPE' ")["H37rv_GeneID"].values )

In [13]:
listOf_13E12_Region_RvIDs = ["Rv0094c", "Rv0095c", "Rv0393", "Rv1572c", "Rv1572c", "Rv1128c", "Rv1148c", "Rv1587c", "Rv1588c", "Rv1702c", "Rv1945", "Rv2100", "Rv3466", "Rv3467"]  


# Parse in homology H37Rv mapping results (k19w19)

In [14]:
Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V9"

H37_Rv_MM2_HomologyMapping_Dir = f"{Main_Project_Dir}/250901.H37Rv.HomologyMapping.k19w19.ProcessedData.V2"

# Define paths to output TSVS

### Homologous regions (MERGED)
RvHmMap_Merged_ParaRegions_TSV  = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.MergedRegions.ParalogousRegions.k19w19.tsv"
RvHmMap_Merged_LocalRepeats_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.MergedRegions.LocalRepeats.k19w19.tsv"

### Homology map (pairwise alignments)
RvHmMap_Aln_All_TSV           = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.All.tsv"
RvHmMap_Aln_PRs_NoOverlap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.NoOverlap.tsv"
RvHmMap_Aln_LRs_WiOverlap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.WiOverlap.tsv"

RvHmMap_Aln_PRs_NoOverlap_Clustered_TSV     = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.NoOverlap.Clustered.tsv"
RvHmMap_Aln_LRs_WiOverlap_Clustered_TSV     = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.Aln.OnlyOverlap.Clustered.tsv"

### Variants from the homology map alignments
RvHmMap_Var_TSV      = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.variants.tsv"
RvHmMap_Var_SNPs_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/RvHmMap.k19w19.variants.snps.tsv"


### Parse in HmRegions (`Paralogous_Regions` and `Local_Repeats`)

In [15]:
HmMapRegs_ParaRegs_k19w19_DF = pd.read_csv(RvHmMap_Merged_ParaRegions_TSV,
                                    sep="\t")

HmMapRegs_ParaRegs_k19w19_DF.shape

(200, 10)

In [16]:
HmMapRegs_LocalRepeats_k19w19_DF = pd.read_csv(RvHmMap_Merged_LocalRepeats_TSV, 
                                        sep="\t")
HmMapRegs_LocalRepeats_k19w19_DF.shape

(50, 10)

In [17]:
HmMapRegs_All_LRsPRs_K19w19_DF = pd.concat([HmMapRegs_ParaRegs_k19w19_DF,
                                            HmMapRegs_LocalRepeats_k19w19_DF])

HmMapRegs_All_LRsPRs_K19w19_DF.shape

(250, 10)

#### Peak at head of each HmMap Regions DFs

In [18]:
HmMapRegs_ParaRegs_k19w19_DF.head()

,HmRegion_Num,Chr,Start,End,Center,Length,Overlap_Genes,NOvrlap_Transposase,HmRegionNum,HmRegionID
0,0,NC_000962.3,80184,80523,80353.5,339,Rv0071,0,0,PR_HmRegion_000
1,1,NC_000962.3,80623,82664,81643.5,2041,"Rv0072,Rv0073",0,1,PR_HmRegion_001
2,2,NC_000962.3,103705,105130,104417.5,1425,"Rv0094c,Rv0095c",0,2,PR_HmRegion_002
3,3,NC_000962.3,149571,149808,149689.5,237,PE_PGRS2,0,3,PR_HmRegion_003
4,4,NC_000962.3,177203,177447,177325.0,244,NaN,0,4,PR_HmRegion_004


In [19]:
HmMapRegs_LocalRepeats_k19w19_DF.head()

,HmRegion_Num,Chr,Start,End,Center,Length,Overlap_Genes,NOvrlap_Transposase,HmRegionNum,HmRegionID
0,0,NC_000962.3,333811,335879,334845.0,2068,PE_PGRS3,0,0,LR_HmRegion_000
1,1,NC_000962.3,366430,375121,370775.5,8691,"PPE5,PPE6",0,1,LR_HmRegion_001
2,2,NC_000962.3,424011,432951,428481.0,8940,"hspR,PPE7,PPE8",0,2,LR_HmRegion_002
3,3,NC_000962.3,566288,580814,573551.0,14526,"hbhA,Rv0476,Rv0477,deoC,Rv0479c,Rv0480c,Rv0481...",0,3,LR_HmRegion_003
4,4,NC_000962.3,631298,631436,631367.0,138,Rv0538,0,4,LR_HmRegion_004


### Parse in homology-map DFs (pairwise alignments between all homologous regions)

In [20]:
HmMap_Aln_k19w19_DF = pd.read_csv(RvHmMap_Aln_All_TSV,
                           sep="\t")
HmMap_Aln_k19w19_DF.shape

(776, 24)

In [21]:
HmMap_Aln_k19w19_NoOverlap_DF = pd.read_csv(RvHmMap_Aln_PRs_NoOverlap_TSV,
                                     sep="\t")
HmMap_Aln_k19w19_NoOverlap_DF.shape

(640, 24)

In [22]:
HmMap_Aln_k19w19_LocalRepeat_DF = pd.read_csv(RvHmMap_Aln_LRs_WiOverlap_TSV,
                                              sep="\t")
HmMap_Aln_k19w19_LocalRepeat_DF.shape

(136, 24)

### Parse in HomologyMap Alignment Variants DFs

In [23]:
Mtb_HM_Var_DF = pd.read_csv(RvHmMap_Var_TSV, sep="\t")
Mtb_HM_Var_SNPs_DF = pd.read_csv(RvHmMap_Var_SNPs_TSV, sep="\t")


In [24]:
Mtb_HM_Var_DF.shape

(79508, 13)

In [25]:
Mtb_HM_Var_SNPs_DF.shape

(65617, 13)

# Define `TBP22-22CI` dataset metadata (Resequenced w/ PacBio HiFi)

### Define file paths to TSVs

In [26]:
Repo_MainDir = "../.."
Repo_DataDir = f"{Repo_MainDir}/Data"

TBP22_GCEVerf_Metadata_Dir = f"{Repo_DataDir}/TBP22.22CI.GCEVerfIsolates.Metadata" 

TBP22_22CI_HybridAsmQCStats_TSV_PATH                = f"{TBP22_GCEVerf_Metadata_Dir}//250801.TBP22.22CI.GCEVerfIsolates.HybridAsmQCStats.V1.tsv"

TBP22_Final_QCPass_22CI_Input_AsmAndReads_Paths_TSV = f"{TBP22_GCEVerf_Metadata_Dir}/250801.TBP22.22CI.GCEVerfIsolates.Asm_LR_SR.InputPATHs.tsv"

TBP22_GCEvent_To_IsolateIDs_TSV = f"{TBP22_GCEVerf_Metadata_Dir}/250801.TBP22.22CI.TGENSR_Reseq.GCEvent_To_IsolateIDs.tsv"

TBP22_IsolateID_To_GCEvents_TSV = f"{TBP22_GCEVerf_Metadata_Dir}/250801.TBP22.22CI.TGENSR_Reseq.IsolateID_To_GCEvents.tsv"


### Read in DFs defining event to isolateID mappings

In [27]:
TBP22_Reseq_GCEvent_To_IsolateIDs_DF = pd.read_csv(TBP22_GCEvent_To_IsolateIDs_TSV, sep="\t" )

# Create EventID → Target dictionary
EventID_to_TargetIsolateID = TBP22_Reseq_GCEvent_To_IsolateIDs_DF.set_index("EventID")["Verification_IsolateID"].to_dict()

# Create EventID → Control dictionary
EventID_to_ControlIsolateID = TBP22_Reseq_GCEvent_To_IsolateIDs_DF.set_index("EventID")["Control_IsolateID"].to_dict()


TBP22_Reseq_GCEvent_To_IsolateIDs_DF.shape

(12, 5)

In [28]:
TBP22_Reseq_GCEvent_To_IsolateIDs_DF

,EventID,Verification_IsolateID,Control_IsolateID,Verfication_SRWGS_RunID,Control_SRWGS_RunID
0,Event_001,TB6733,TB3898,SRR10379945,SRR10397263
1,Event_003,TB6599,TB6977,SRR10379958,SRR10380218
2,Event_006,TB3305,TB3706,SRR10397175,SRR10397096
3,Event_007,TB6755,TB7044,SRR10379935,SRR10380192
4,Event_010,TB6552,TB6765,SRR10380108,SRR10379924
5,Event_011,TB6778,TB6973,SRR10380252,SRR10380223
6,Event_013,TB6786,TB3256,SRR10380244,SRR10397205
7,Event_019,TB6977,TB6976,SRR10380218,SRR10380219
8,Event_021,TB3572,TB6976,SRR10397163,SRR10380219
9,Event_022,TB6596,TB8073,SRR10379961,SRR10380026


### Read in DFs defining isolateID eventID mappings

In [29]:
TBP22_Reseq_IsolateID_To_GCEvents_DF = pd.read_csv(TBP22_IsolateID_To_GCEvents_TSV, sep="\t" )

TBP22_Reseq_IsolateID_To_GCEvents_DF.shape

(22, 4)

### TBP22 Hybrid Complete Genome Assembly QC Stats 

In [30]:
TBP22_22CI_AsmQC_DF = pd.read_csv(TBP22_22CI_HybridAsmQCStats_TSV_PATH,
                                  sep ="\t")
TBP22_22CI_AsmQC_DF.shape

(22, 19)

In [31]:
TBP22_22CI_AsmQC_DF.head()

,SampleID,numContigs_Complete,circContig_Length,circContig_Cov,Flye_EstimatedCov,Flye_ReadLen_N50,Flye_ReadLen_N90,Lineage_Asm,Lineage_AsmPP,PrimaryLineage_Asm,Dataset_Tag,SR_SRA_RunAcc,SeqReason,PB_SeqRunName,EventID,Event_Gene(s),Event_Relationship,TBP_SampleID,TGEN_SampleID
0,TB3706,1,4412093,127,129,4204,2724,lineage2.2.1,lineage2.2.1,lineage2,TBPortals2022,SRR10397096,ReseqToVerfGCE,P7529,Event_006,"Rv0979c,rpmF,PE_PGRS18",Outside_Event,TB3706,DNA621
1,TB3305,1,4416251,161,174,4212,2712,lineage2.2.1,lineage2.2.1,lineage2,TBPortals2022,SRR10397175,ReseqToVerfGCE,P7529,Event_006,"Rv0979c,rpmF,PE_PGRS18",Within_Event,TB3305,DNA594
2,TB6755,1,4417503,307,309,7792,6519,lineage4.1.2.1,lineage4.1.2.1,lineage4,TBPortals2022,SRR10379935,ReseqToVerfGCE,P7559,Event_007,Rv1148c,Within_Event,TB6755,DNA0432
3,TB6552,1,4408536,151,151,8683,7029,lineage4.1.2.1,lineage4.1.2.1,lineage4,TBPortals2022,SRR10380108,ReseqToVerfGCE,P7544,Event_010,"PPE18,esxK,esxL",Within_Event,TB6552,DNA199
4,TB6765,1,4386061,82,82,8459,6983,lineage4.1.2.1,lineage4.1.2.1,lineage4,TBPortals2022,SRR10379924,ReseqToVerfGCE,P7544,Event_010,"PPE18,esxK,esxL",Outside_Event,TB6765,DNA0441


### Define dictionaries that map sampleID to metadata labels

In [32]:
TBP22_ID_To_PrimLineage_Dict = dict( TBP22_22CI_AsmQC_DF[['TBP_SampleID', 'PrimaryLineage_Asm']].values)
TBP22_ID_To_SubLineage_Dict  = dict( TBP22_22CI_AsmQC_DF[["TBP_SampleID", "Lineage_AsmPP"]].values)


### TBP22 - Assembly + Illumina WGS + PacBio WGS File Paths

In [33]:
TBP22_22CI_EventVerf_AsmAndRead_PATHS_V1_DF = pd.read_csv(TBP22_Final_QCPass_22CI_Input_AsmAndReads_Paths_TSV,
                                                                  sep ="\t")


TBP22_22CI_SampleIDs = TBP22_22CI_EventVerf_AsmAndRead_PATHS_V1_DF["SampleID"].values

TBP22_22CI_EventVerf_AsmAndRead_PATHS_V1_DF.shape

(22, 8)

In [34]:
TBP22_22CI_SampleIDs

array(['TB6733', 'TB3898', 'TB7340', 'TB6977', 'TB3305', 'TB3706',
       'TB6755', 'TB7044', 'TB6765', 'TB6552', 'TB6778', 'TB6973',
       'TB6786', 'TB3256', 'TB4414', 'TB6599', 'TB6976', 'TB3572',
       'TB6596', 'TB8073', 'TB6807', 'TB6846'], dtype=object)

# Copy `TBP22-22CI` Gubbins output from SMK output dir to analysis dir

### Define directories of the assembly + analysis pipeline 

In [35]:
### Define directories to PMP-SM (PacBio assembly and analysis pipeline)

### Define PATH to pipeline output directories

Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects"

Mtb_WGA_SMK_Outputs_Dir = Project_Dir + "/Mtb-WGA-SMK-Output"

TBP22_22CI_SMK_OutputDir = Mtb_WGA_SMK_Outputs_Dir + "/250730_TBP22_22CI_V2"


In [36]:
!ls -1 $TBP22_22CI_SMK_OutputDir | head -n 10

AsmAnalysis
Asm_MergeSNPs_mpileup
Asm_MergeVar_mpileup
benchmarks
HomologyMapping
NucDiversity
Phylogenies
RecombDetection
slurm_logs
TB3256


### Define output sub-directories of pipeline

In [37]:
target_OutputDir = TBP22_22CI_SMK_OutputDir

i_Phylogeny_Dir = f"{target_OutputDir}/Phylogenies"

i_NucDiv_Dir = f"{target_OutputDir}/NucDiversity"
i_NucDiv_mpileup_Dir = f"{i_NucDiv_Dir}/NucDiv_SNVs_mpileup"

i_Recomb_Dir = f"{target_OutputDir}/RecombDetection"
i_Gubbins_Out_Dir = f"{i_Recomb_Dir}/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh"



In [38]:
!ls -1 $i_Recomb_Dir

Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh
PopulationMSA_mpileup_SNVs_10AmbThresh


In [39]:
!ls -1 $i_Gubbins_Out_Dir

Gubbins.branch_base_reconstruction.embl
Gubbins.filtered_polymorphic_sites.fasta
Gubbins.filtered_polymorphic_sites.phylip
Gubbins.final_tree.tre
Gubbins.log
Gubbins.node_labelled.final_tree.tre
Gubbins.per_branch_statistics.csv
Gubbins.recombination_predictions.embl
Gubbins.recombination_predictions.gff
Gubbins.recombination_predictions.RenamedCHR.bed
Gubbins.recombination_predictions.RenamedCHR.gff
Gubbins.summary_of_snp_distribution.vcf


# 1) Define relevant PATHS and target directories

In [40]:
AnalysisName = "250801.TBP22_22CI.V1"

Gubbins_OutPrefix = "Gubbins"

Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb-GeneConv"

!mkdir $Main_Project_Dir

Target_Output_Dir = f"{Main_Project_Dir}/{AnalysisName}"

!mkdir $Target_Output_Dir

input_SampleNames = TBP22_22CI_SampleIDs


mkdir: cannot create directory ‘/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb-GeneConv’: File exists
mkdir: cannot create directory ‘/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb-GeneConv/250801.TBP22_22CI.V1’: File exists


## Copy Gubbins output directory (From SMK pipeline output) to analysis directory

In [41]:
!cp -r $i_Gubbins_Out_Dir/ $Target_Output_Dir/

In [42]:
!ls -alh $i_Gubbins_Out_Dir

total 2.0M
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Jul 30 17:44 .
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Jul 30 17:42 ..
-rw-r--r-- 1 mm774 hpc_farhat 1.4M Jul 30 17:43 Gubbins.branch_base_reconstruction.embl
-rw-r--r-- 1 mm774 hpc_farhat 100K Jul 30 17:43 Gubbins.filtered_polymorphic_sites.fasta
-rw-r--r-- 1 mm774 hpc_farhat 100K Jul 30 17:43 Gubbins.filtered_polymorphic_sites.phylip
-rw-r--r-- 1 mm774 hpc_farhat  631 Jul 30 17:43 Gubbins.final_tree.tre
-rw-r--r-- 1 mm774 hpc_farhat  491 Jul 30 17:43 Gubbins.log
-rw-r--r-- 1 mm774 hpc_farhat  777 Jul 30 17:43 Gubbins.node_labelled.final_tree.tre
-rw-r--r-- 1 mm774 hpc_farhat 2.8K Jul 30 17:43 Gubbins.per_branch_statistics.csv
-rw-r--r-- 1 mm774 hpc_farhat  22K Jul 30 17:43 Gubbins.recombination_predictions.embl
-rw-r--r-- 1 mm774 hpc_farhat  13K Jul 30 17:43 Gubbins.recombination_predictions.gff
-rw-r--r-- 1 mm774 hpc_farhat  13K Jul 30 17:44 Gubbins.recombination_predictions.RenamedCHR.bed
-rw-r--r-- 1 mm774 hpc_farhat  13K Jul 30 17:44 Gubbi

## Define paths to COPIED GUBBINS outputs

In [43]:
Gubbins_V1_NEW_ResultsDir = f"{Target_Output_Dir}/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh"

In [44]:
Gubbins_NodeLabelledTree_PATH      = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.node_labelled.final_tree.tre"

Gubbins_BranchStats_CSV_PATH       = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.per_branch_statistics.csv"


Gubbins_RecombPreds_GFF            = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.recombination_predictions.gff"
Gubbins_RecombPreds_EMBL           = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.recombination_predictions.embl"

Gubbins_RecombPreds_RenamedChr_GFF = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.recombination_predictions.RenamedCHR.gff"
Gubbins_RecombPreds_RenamedChr_BED = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.recombination_predictions.RenamedCHR.bed"      

Gubbins_BaseReconstruction_EMBL    = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.branch_base_reconstruction.embl"


In [45]:
print(Gubbins_V1_NEW_ResultsDir)

/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb-GeneConv/250801.TBP22_22CI.V1/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh


In [46]:
!ls -alh $Gubbins_V1_NEW_ResultsDir

total 3.3M
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Aug  2 20:49 .
drwxr-sr-x 2 mm774 hpc_farhat 4.0K Aug  1 17:13 ..
-rw-r--r-- 1 mm774 hpc_farhat 411K Aug  3 15:51 Gubbins.branch_base_reconstruction.AnnoByEvent.All.tsv
-rw-r--r-- 1 mm774 hpc_farhat  62K Aug  3 15:51 Gubbins.branch_base_reconstruction.AnnoByEvent.EventSNPsOnly.tsv
-rw-r--r-- 1 mm774 hpc_farhat 1.4M Sep  1 16:25 Gubbins.branch_base_reconstruction.embl
-rw-r--r-- 1 mm774 hpc_farhat 100K Sep  1 16:25 Gubbins.filtered_polymorphic_sites.fasta
-rw-r--r-- 1 mm774 hpc_farhat 100K Sep  1 16:25 Gubbins.filtered_polymorphic_sites.phylip
-rw-r--r-- 1 mm774 hpc_farhat  631 Sep  1 16:25 Gubbins.final_tree.tre
-rw-r--r-- 1 mm774 hpc_farhat 263K Aug  3 15:51 Gubbins.H37Rv.EventsPer1kb.tsv
-rw-r--r-- 1 mm774 hpc_farhat 600K Aug  3 15:51 Gubbins.H37Rv.EventsPerGene.tsv
-rw-r--r-- 1 mm774 hpc_farhat  24K Aug  3 15:51 Gubbins.H37Rv.EventsPerMergedHomologousRegion.tsv
-rw-r--r-- 1 mm774 hpc_farhat  491 Sep  1 16:25 Gubbins.log
-rw-r--r-- 1 mm77

# Begin Processing of Gubbins Results

# Gubbins Phylogeny Processing

## 1) Parse Gubbins Tree w/ ETE3

In [47]:
i_Gubbins_T = Tree(Gubbins_NodeLabelledTree_PATH, format = 1)


## 2) Parsing over each node of the tree (ETE3), get ML BRANCH LENGTH and Number of descendents per node

In [48]:
Tree_BranchLen_Dict, NumDescendants_PerLeaf_Dict = get_BranchLengths_and_DescendantCounts_FromTree(i_Gubbins_T)

In [49]:
print( len(list(Tree_BranchLen_Dict.keys())) )

42


In [50]:
print( len(list(NumDescendants_PerLeaf_Dict.keys())) )

42


## 3) Label each LEAF within phylogeny by isolate's MTBC Lineage

In [51]:
count = 0
for n in i_Gubbins_T.get_leaves():
    n.add_feature("Primary_lineage", TBP22_ID_To_PrimLineage_Dict.get(n.name, "Unknown Lineage") )
    n.add_feature("Sublineage", TBP22_ID_To_SubLineage_Dict.get(n.name, "Unknown Lineage") )
    #print("node:", n.name, " Lineage:", n.Mtb_lineage)
    count +=1
    
print(count)
    
i_Gubbins_T.sort_descendants(attr='Primary_lineage')


22


## 4) Infer lineage each node of the tree (ETE3)

In [52]:
node_To_PrimaryLin_Dict = {}

for node in i_Gubbins_T.iter_descendants("postorder"):
    
    listOf_ChildLineages = []
    
    for child_node in node.get_descendants():
        if child_node.is_leaf():
            listOf_ChildLineages.append(  (child_node.Primary_lineage) )
                #print(node.name, listOf_ChildLineages)
        
    set_Of_ChildLineages = list(set(listOf_ChildLineages))
    
    if len(set_Of_ChildLineages) == 1:
        OnlyOneLineage = True
    else:
        OnlyOneLineage = False
    
    if OnlyOneLineage:
        node_To_PrimaryLin_Dict[node.name] = set_Of_ChildLineages[0]

node_To_PrimaryLin_Dict.update(TBP22_ID_To_PrimLineage_Dict)
    

In [53]:
child_node.name

'TB8073'

In [54]:
child_node.Primary_lineage

'lineage4'

## 5) Create a mapping of the primary lineage of each NODE to SampleID

### Output "node_To_PrimaryLin_Dict" dictionary 

In [55]:
Gubbins_NodeToPriLineage_Dict_JSON = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.NodeToPrimaryLineage.json"

with open(Gubbins_NodeToPriLineage_Dict_JSON, 'w') as json_file:
    json.dump(node_To_PrimaryLin_Dict, json_file)


#### test reading back in the JSON

In [56]:
with open(Gubbins_NodeToPriLineage_Dict_JSON) as json_file:
    node_To_PrimaryLin_Dict = json.load(json_file)

In [57]:
len(list(node_To_PrimaryLin_Dict.keys()))

42

In [58]:
list(node_To_PrimaryLin_Dict.keys())[:5]

['Node_1', 'Node_12', 'Node_13', 'Node_14', 'Node_15']

In [59]:
node_To_PrimaryLin_Dict["Node_7"]

'lineage4'

In [60]:
#node_To_PrimaryLin_Dict

# Parse & Annotate Gubbins Recombination GFFs (Individual Events)

### This gives us info at the individual event level (inferred by Gubbins)

In [61]:
Gubbins_Recomb_Events_DF = get_RecombEvents_From_Gubbins_GFF_V2(Gubbins_RecombPreds_RenamedChr_GFF)

Gubbins_Recomb_Events_DF["Lineage"] = Gubbins_Recomb_Events_DF["Child_Node"].map(node_To_PrimaryLin_Dict).fillna("None")

Gubbins_Recomb_Events_DF.shape

(84, 14)

In [62]:
H37Rv_GenomeAnno_Genes_DF.head(3)

,Chrom,Start,End,Strand,H37rv_GeneID,Symbol,Feature,Functional_Category,Is_Pseudogene,Product,PEandPPE_Subfamily,ExcludedGroup_Category
0,NC_000962.3,0,1524,+,Rv0001,dnaA,CDS,information pathways,No,Chromosomal replication initiator protein DnaA,NaN,NotExcluded
1,NC_000962.3,2051,3260,+,Rv0002,dnaN,CDS,information pathways,No,DNA polymerase III (beta chain) DnaN (DNA nucl...,NaN,NotExcluded
2,NC_000962.3,3279,4437,+,Rv0003,recF,CDS,information pathways,No,DNA replication and repair protein RecF (singl...,NaN,NotExcluded


## Label Gubbins Recomb Events by overlapping genes

In [63]:
print(Gubbins_Recomb_Events_DF.shape)
Gubbins_Recomb_Events_DF = label_Gubbins_Events_DF_ByOvrLap_H37RvGenes(Gubbins_Recomb_Events_DF, 
                                                                       H37Rv_GenomeAnno_Genes_DF)
print(Gubbins_Recomb_Events_DF.shape)

(84, 14)
(84, 16)


## Let's label each putative recombination event by whether it overlaps with one of the 3 gene-groups (Esx, PE/PPE, REP13E12 repeats, or None)

In [64]:
print("# of Esx genes:", len(ListOf_Esx_RvIDs))
print("# of PE/PPE genes:", len(listOf_PEPPE_RvIDs))
print("# of 13E12 Repeat Region genes:", len(listOf_13E12_Region_RvIDs))

# of Esx genes: 23
# of PE/PPE genes: 168
# of 13E12 Repeat Region genes: 14


In [65]:
# Apply the overlap function
G = Gubbins_Recomb_Events_DF.copy()
G["Contains_Esx"] = G["Overlap_Gene_RvIDs"].apply(lambda x: check_overlap_with_gene_group(x, ListOf_Esx_RvIDs))
G["Contains_PEPPE"] = G["Overlap_Gene_RvIDs"].apply(lambda x: check_overlap_with_gene_group(x, listOf_PEPPE_RvIDs))
G["Contains_REP13E12"] = G["Overlap_Gene_RvIDs"].apply(lambda x: check_overlap_with_gene_group(x, listOf_13E12_Region_RvIDs))
G["NoOverlap_Wi_PEPPE_Esx_13E12_Genes"] = ~(G["Contains_Esx"] | G["Contains_PEPPE"] | G["Contains_REP13E12"] )
Gubbins_Recomb_Events_DF = G.copy()

#### Peak at stats of overlap with gene groups commonly effected by gene conversion

In [66]:
Gubbins_Recomb_Events_DF["Contains_Esx"].value_counts()

Contains_Esx
False    76
True      8
Name: count, dtype: int64

In [67]:
Gubbins_Recomb_Events_DF["Contains_PEPPE"].value_counts()

Contains_PEPPE
True     45
False    39
Name: count, dtype: int64

In [68]:
Gubbins_Recomb_Events_DF["Contains_REP13E12"].value_counts()

Contains_REP13E12
False    64
True     20
Name: count, dtype: int64

In [69]:
Gubbins_Recomb_Events_DF["NoOverlap_Wi_PEPPE_Esx_13E12_Genes"].value_counts()

NoOverlap_Wi_PEPPE_Esx_13E12_Genes
False    73
True     11
Name: count, dtype: int64

## Generate EventIDs for all detected events

In [70]:
## 1) Create ordered version of all putative recombination events.
## - Let's sort by the following columns: ["start_1based", "end_1based", "Parent_Node", "Child_Node", "Lineage", "Overlap_Genes"]

Gub_ColumnsToSortBy = ["start_1based", "end_1based", "Parent_Node", "Child_Node", "Lineage", "Overlap_Genes"]

Gubbins_Recomb_Events_DF = Gubbins_Recomb_Events_DF.sort_values(Gub_ColumnsToSortBy, kind="mergesort").reset_index(drop=True)

Gubbins_Recomb_Events_DF["EventNum"] = Gubbins_Recomb_Events_DF.index + 1
Gubbins_Recomb_Events_DF["EventID"] = "Event_" + Gubbins_Recomb_Events_DF["EventNum"].astype(str).str.rjust(3, '0')

Gubbins_Recomb_Events_DF = Gubbins_Recomb_Events_DF.drop("EventNum", axis = 1)


In [71]:
Gubbins_Recomb_Events_DF["EventLen"].describe()

count      84.000000
mean      261.892857
std       314.687453
min         7.000000
25%        47.750000
50%       131.500000
75%       323.500000
max      1270.000000
Name: EventLen, dtype: float64

In [72]:
Gubbins_Recomb_Events_DF["neg_log_likelihood"].describe()

count      84.000000
mean     1078.334186
std       897.531346
min        62.713784
25%       620.756320
50%       862.176305
75%      1070.654350
max      3789.314751
Name: neg_log_likelihood, dtype: float64

In [73]:
Gubbins_Recomb_Events_DF.shape

(84, 21)

In [74]:
Gubbins_Recomb_Events_DF.head(2)

,seqname,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID
0,NC_000962.3,103756,103896,0.000,.,Node_8,TB6977,876.132298,4,[TB6977],103755,103825.5,141,lineage4,Rv0094c,Rv0094c,False,False,True,False,Event_001
1,NC_000962.3,103776,105045,0.000,.,Node_20,Node_3,1661.496090,14,"[TB6552, TB6755, TB6765]",103775,104410.0,1270,lineage4,"Rv0094c,Rv0095c","Rv0094c,Rv0095c",False,False,True,False,Event_002


### Annotate each event by whether an HHR overlaps

In [1]:
RE_CoordCols = ("seqname", "start_0based", "end_1based")
HmAlnPAF_CoordCols = ("Query_Name", "Query_Start", "Query_End")
HHR_CoordCols = ("Chr", "Start", "End")

#### A
Gubbins_Recomb_Events_DF = bf.count_overlaps(Gubbins_Recomb_Events_DF,
                                             HmMap_Aln_k19w19_NoOverlap_DF, 
                                             cols1 = RE_CoordCols,
                                             cols2 = HmAlnPAF_CoordCols ).rename(columns={'count': 'N_HmMapAln_PR_Ovrlap'})


Gubbins_Recomb_Events_DF["OvrlapWi_HmMapAln_PR"] = np.where(Gubbins_Recomb_Events_DF['N_HmMapAln_PR_Ovrlap'] > 0, 1, 0)


#### B
Gubbins_Recomb_Events_DF = bf.count_overlaps(Gubbins_Recomb_Events_DF,
                                             HmMap_Aln_k19w19_LocalRepeat_DF, 
                                             cols1 = RE_CoordCols,
                                             cols2 = HmAlnPAF_CoordCols ).rename(columns={'count': 'N_HmMapAln_LR_Ovrlap'})


Gubbins_Recomb_Events_DF["OvrlapWi_HmMapAln_LR"] = np.where(Gubbins_Recomb_Events_DF['N_HmMapAln_LR_Ovrlap'] > 0, 1, 0)



NameError: name 'bf' is not defined

In [ ]:
Gubbins_Recomb_Events_DF.shape

In [77]:
#Gubbins_Recomb_Events_DF.head()

### Annotate each event by the overlapping HHRs (Each HHR ID)

In [78]:
Gubbins_Recomb_Events_DF = label_Gubbins_Events_DF_ByOvrLap_HHRs(Gubbins_Recomb_Events_DF,
                                                                 HmMapRegs_All_LRsPRs_K19w19_DF)    

In [79]:
Gubbins_Recomb_Events_DF[Gubbins_Recomb_Events_DF["Overlap_HHRs"].isna()]

,seqname,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,N_HmMapAln_PR_Ovrlap,OvrlapWi_HmMapAln_PR,N_HmMapAln_LR_Ovrlap,OvrlapWi_HmMapAln_LR,Overlap_HHRs


In [80]:
Gubbins_Recomb_Events_DF[Gubbins_Recomb_Events_DF["Overlap_HHRs"] == '']

,seqname,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,N_HmMapAln_PR_Ovrlap,OvrlapWi_HmMapAln_PR,N_HmMapAln_LR_Ovrlap,OvrlapWi_HmMapAln_LR,Overlap_HHRs
53,NC_000962.3,2338866,2338994,0.000,.,Node_19,Node_7,1124.033576,5,"[TB6778, TB6973, TB7044, TB7340, TB6807]",2338865,2338929.5,129,lineage4,Rv2082,Rv2082,False,False,False,True,Event_054,0,0,0,0,
60,NC_000962.3,3054702,3054717,0.000,.,Node_8,TB6977,907.431410,5,[TB6977],3054701,3054709.0,16,lineage4,PE_PGRS47,Rv2741,False,True,False,False,Event_061,0,0,0,0,
62,NC_000962.3,3477266,3477370,0.000,.,Node_9,TB3572,832.925610,5,[TB3572],3477265,3477317.5,105,lineage4,Rv3108,Rv3108,False,False,False,True,Event_063,0,0,0,0,


In [82]:
Gubbins_Recomb_Events_DF.shape

(84, 26)

In [83]:
Gubbins_Recomb_Events_DF.shape

(84, 26)

In [84]:
Gubbins_Recomb_Events_DF["Overlap_HHRs"].value_counts().head(3)

Overlap_HHRs
PR_HmRegion_002    11
PR_HmRegion_065     5
PR_HmRegion_179     4
Name: count, dtype: int64

## Output updated Gubbins_Recomb_Events_DF to TSV

In [85]:
Gubbins_RecombPreds_Anno_TSV = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.recombination_predictions.Anno.tsv"

Gubbins_Recomb_Events_DF.to_csv(Gubbins_RecombPreds_Anno_TSV, sep="\t", index=False)


### Explore a bit the detected GC events in TGEN dataset

In [86]:
Gubbins_Recomb_Events_DF.head(2)

,seqname,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,N_HmMapAln_PR_Ovrlap,OvrlapWi_HmMapAln_PR,N_HmMapAln_LR_Ovrlap,OvrlapWi_HmMapAln_LR,Overlap_HHRs
0,NC_000962.3,103756,103896,0.000,.,Node_8,TB6977,876.132298,4,[TB6977],103755,103825.5,141,lineage4,Rv0094c,Rv0094c,False,False,True,False,Event_001,2,1,0,0,PR_HmRegion_002
1,NC_000962.3,103776,105045,0.000,.,Node_20,Node_3,1661.496090,14,"[TB6552, TB6755, TB6765]",103775,104410.0,1270,lineage4,"Rv0094c,Rv0095c","Rv0094c,Rv0095c",False,False,True,False,Event_002,2,1,0,0,PR_HmRegion_002


In [87]:
Gubbins_Recomb_Events_DF["Overlap_Genes"].value_counts().head(20)

Overlap_Genes
Rv0095c      9
PE_PGRS28    5
PPE54        5
Rv1588c      4
PE_PGRS27    4
PPE19        3
Rv1148c      3
PE_PGRS54    3
PPE60        3
PE_PGRS4     3
PPE59        3
esxI,esxJ    2
esxV,esxW    2
PPE55        2
Rv2177c      2
esxK,esxL    2
PE_PGRS18    2
PE_PGRS17    2
PPE25        2
Rv0750       1
Name: count, dtype: int64

#### How many GC events DO occur in a HHR? (79/84)

In [88]:
Gubbins_Recomb_Events_DF.shape

(84, 26)

In [90]:
Gubbins_Recomb_Events_DF.query("N_HmMapAln_PR_Ovrlap > 0").shape

(79, 26)

In [91]:
Gubbins_Recomb_Events_DF.query("N_HmMapAln_PR_Ovrlap > 0 | N_HmMapAln_LR_Ovrlap > 0").shape

(81, 26)

#### Which GC events do NOT occur in a HHR? (5/84)

In [92]:
Gubbins_Recomb_Events_DF.query("N_HmMapAln_PR_Ovrlap == 0 & N_HmMapAln_LR_Ovrlap == 0").shape

(3, 26)

In [94]:
Gubbins_Recomb_Events_DF.query("N_HmMapAln_PR_Ovrlap == 0 & N_HmMapAln_LR_Ovrlap == 0")["Overlap_Genes"].values


array(['Rv2082', 'PE_PGRS47', 'Rv3108'], dtype=object)

# Count recomb events per region

### Count inferred recombination events (Gubbins) per: <br> 
a) 1 kb region <br> 
b) gene <br> 
c) merged-homologous region <br>

## Read in annotated windows of the H37Rv genome

In [95]:
RepoRef_Dir = "../../References"

H37Rv_Windows_Dir = f"{RepoRef_Dir}/H37Rv_GenomeWindows"

H37Rv_1kb_Win_Anno_TSV = f"{H37Rv_Windows_Dir}/H37Rv.1000bp.Windows.Anno.tsv"

Rv_1kb_Win_DF = pd.read_csv(H37Rv_1kb_Win_Anno_TSV, sep = "\t")
Rv_1kb_Window_Start_To_Genes_Dict = dict(Rv_1kb_Win_DF[['Start', 'Overlap_Genes']].values)


### a) Let's count events per 1 kb window

In [96]:
RvWin_CoordCols = ("Chrom", "Start", "End")
RE_CoordCols = ("seqname", "start_0based", "end_1based")

Rv_1kb_RE_Count_DF = bf.count_overlaps(Rv_1kb_Win_DF,
                                       Gubbins_Recomb_Events_DF,
                                       cols1 = RvWin_CoordCols,
                                       cols2 = RE_CoordCols)

Rv_1kb_RE_Count_DF.rename(columns={'count': 'pGCE_Count'}, inplace=True)

Rv_1kb_RE_Count_DF["CenterOfRegion"] = ((Rv_1kb_RE_Count_DF["Start"] + Rv_1kb_RE_Count_DF["End"]) / 2)

Rv_1kb_RE_Count_DF.shape

(4412, 7)

In [97]:
Rv_1kb_RE_Count_DF.sort_values("pGCE_Count", ascending=False).head(4)

,Chrom,Start,End,Overlap_Genes,Middle,pGCE_Count,CenterOfRegion
104,NC_000962.3,104000,105000,"Rv0094c,Rv0095c",104500.0,10,104500.0
105,NC_000962.3,105000,106000,"Rv0095c,PPE1",105500.0,5,105500.0
1638,NC_000962.3,1638000,1639000,"PE_PGRS28,Rv1453",1638500.0,5,1638500.0
3847,NC_000962.3,3847000,3848000,"PPE59,Rv3430c",3847500.0,4,3847500.0


### d) Count events per gene

In [98]:
RE_CoordCols = ("seqname", "start_0based", "end_1based")
GenomeAnno_CoordCols = ("Chrom", "Start", "End")

Rv_Genes_RE_Count_DF = bf.count_overlaps(H37Rv_GenomeAnno_Genes_DF,
                                         Gubbins_Recomb_Events_DF,
                                         cols1 = GenomeAnno_CoordCols,
                                         cols2 = RE_CoordCols)

Rv_Genes_RE_Count_DF.rename(columns={'count': 'pGCE_Count'}, inplace=True)
Rv_Genes_RE_Count_DF["CenterOfRegion"] = ((Rv_Genes_RE_Count_DF["Start"] + Rv_Genes_RE_Count_DF["End"]) / 2)

Rv_Genes_RE_Count_DF.shape

(4079, 14)

### e) Count events per HHR (MERGED homologous region)

In [99]:
RE_CoordCols = ("seqname", "start_0based", "end_1based")
HmRegion_CoordCols = ("Chr", "Start", "End")

Rv_HHRs_RE_Count_DF = bf.count_overlaps(HmMapRegs_ParaRegs_k19w19_DF,
                                        Gubbins_Recomb_Events_DF,
                                        cols1 = HmRegion_CoordCols,
                                        cols2 = RE_CoordCols)

Rv_HHRs_RE_Count_DF["CenterOfRegion"] = ((Rv_HHRs_RE_Count_DF["Start"] + Rv_HHRs_RE_Count_DF["End"]) / 2)

Rv_HHRs_RE_Count_DF.rename(columns={'count': 'pGCE_Count'}, inplace=True)

#### Annotate HHRs by the eventIDs that overlap
Rv_HHRs_RE_Count_DF = annotate_HHR_with_GCE_overlaps(Rv_HHRs_RE_Count_DF,
                                                     Gubbins_Recomb_Events_DF)


Rv_HHRs_RE_Count_DF.shape

(200, 13)

#### Peak at HHR level summary of pGCEs

In [100]:
Rv_HHRs_RE_Count_DF.head(2)

,HmRegion_Num,Chr,Start,End,Center,Length,Overlap_Genes,NOvrlap_Transposase,HmRegionNum,HmRegionID,pGCE_Count,CenterOfRegion,Overlap_GC_EventIDs
0,0,NC_000962.3,80184,80523,80353.5,339,Rv0071,0,0,PR_HmRegion_000,0,80353.5,None
1,1,NC_000962.3,80623,82664,81643.5,2041,"Rv0072,Rv0073",0,1,PR_HmRegion_001,0,81643.5,None


In [101]:
Rv_HHRs_RE_Count_DF.sort_values("pGCE_Count", ascending=False).head(1)

,HmRegion_Num,Chr,Start,End,Center,Length,Overlap_Genes,NOvrlap_Transposase,HmRegionNum,HmRegionID,pGCE_Count,CenterOfRegion,Overlap_GC_EventIDs
2,2,NC_000962.3,103705,105130,104417.5,1425,"Rv0094c,Rv0095c",0,2,PR_HmRegion_002,11,104417.5,"Event_001,Event_002,Event_003,Event_004,Event_..."


### Output pGCE counts per region TSVs

In [102]:
Gubbins_EventsPer_1kb_H37Rv_TSV_PATH                = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.H37Rv.EventsPer1kb.tsv"

Gubbins_EventsPer_Gene_H37Rv_TSV_PATH               = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.H37Rv.EventsPerGene.tsv"
Gubbins_EventsPer_MgHomologousRegion_H37Rv_TSV_PATH = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.H37Rv.EventsPerMergedHomologousRegion.tsv"

Rv_1kb_RE_Count_DF.to_csv(Gubbins_EventsPer_1kb_H37Rv_TSV_PATH, sep="\t", index=False)
Rv_Genes_RE_Count_DF.to_csv(Gubbins_EventsPer_Gene_H37Rv_TSV_PATH, sep="\t", index=False)
Rv_HHRs_RE_Count_DF.to_csv(Gubbins_EventsPer_MgHomologousRegion_H37Rv_TSV_PATH, sep="\t", index=False)


# Parse base reconstruction of Gubbins (EMBL)

In [103]:
Gubbins_SNPs_All_DF = parse_BaseReconstruction_EMBL_Gubbins(Gubbins_BaseReconstruction_EMBL)
Gubbins_SNPs_All_DF.shape

(5814, 6)

In [104]:
Gubbins_SNPs_All_DF.head()

,Pos_1based,Parent_Node,Child_Node,Parent_Call,Child_Call,taxa_List
0,70116,Node_1,TB3305,C,T,TB3305
1,112554,Node_1,TB3305,G,A,TB3305
2,132649,Node_1,TB3305,G,T,TB3305
3,178624,Node_1,TB3305,G,A,TB3305
4,196891,Node_1,TB3305,G,T,TB3305


In [105]:
Gubbins_SNPs_All_DF.shape

(5814, 6)

# Label all SNP mutational events by associated EventID

In [106]:
Gubbins_SNPs_All_Anno_DF = annotate_Gubbins_SNP_Events_By_RecombEventID(Gubbins_SNPs_All_DF, Gubbins_Recomb_Events_DF)
Gubbins_SNPs_All_Anno_DF.shape

(5814, 7)

In [107]:
Gubbins_SNPs_EventOnly_Anno_DF = Gubbins_SNPs_All_Anno_DF.query("EventID != 'None'")
Gubbins_SNPs_EventOnly_Anno_DF.shape

(966, 7)

In [108]:
Gubbins_SNPs_All_Anno_DF["EventID"].value_counts().head(10)

EventID
None         4848
Event_032      52
Event_077      51
Event_016      47
Event_026      42
Event_079      29
Event_075      29
Event_073      27
Event_076      25
Event_004      24
Name: count, dtype: int64

In [109]:
Gubbins_SNPs_All_Anno_DF.head(3)

,Pos_1based,Parent_Node,Child_Node,Parent_Call,Child_Call,taxa_List,EventID
0,1094228,Node_1,TB3305,T,G,TB3305,Event_019
1,1094375,Node_1,TB3305,C,G,TB3305,Event_019
2,1094538,Node_1,TB3305,T,G,TB3305,Event_019


In [110]:
Gubbins_SNPs_All_Anno_DF.tail(3)

,Pos_1based,Parent_Node,Child_Node,Parent_Call,Child_Call,taxa_List,EventID
5811,4374736,Node_9,TB6976,C,G,TB6976,None
5812,4380656,Node_9,TB6976,G,A,TB6976,None
5813,4380753,Node_9,TB6976,C,G,TB6976,None


In [111]:
Gubbins_SNPs_EventOnly_Anno_DF.shape

(966, 7)

In [112]:
Gubbins_SNPs_All_Anno_DF.shape

(5814, 7)

### How many total SNPs associated with ALL EVENTS detected?

Answer: Agreement between the "GC events DF" and the "SNPs DF"

In [113]:
Gubbins_Recomb_Events_DF["snp_count"].sum()

966

In [114]:
Gubbins_SNPs_EventOnly_Anno_DF.shape[0]

966

## Output Gubbins' Base-Reconstruction SNP dataframe (Annotated)

In [115]:
Gubbins_BaseReconstruction_Anno_TSV = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.branch_base_reconstruction.AnnoByEvent.All.tsv"   
Gubbins_BaseReconstruction_Anno_EventOnly_TSV = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.branch_base_reconstruction.AnnoByEvent.EventSNPsOnly.tsv"   


In [116]:
Gubbins_SNPs_All_Anno_DF.to_csv(Gubbins_BaseReconstruction_Anno_TSV, sep="\t", index=False)

In [117]:
Gubbins_SNPs_EventOnly_Anno_DF.to_csv(Gubbins_BaseReconstruction_Anno_EventOnly_TSV, sep="\t", index=False)

# Process the Gubbins' Branch stats

In [118]:
G_BranchStats_DF = pd.read_csv(Gubbins_BranchStats_CSV_PATH, sep = "\t")
G_BranchStats_DF.shape

(43, 11)

In [119]:
G_BranchStats_DF.head()

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame
0,TB6733,164,11,153,1,333,2923,0.071895,0.006536,4411490,4409032
1,TB3898,48,5,43,1,333,2612,0.116279,0.023256,4411532,4409167
2,TB7340,34,29,5,1,351,3069,5.800000,0.200000,4411532,4408587
3,TB6977,193,13,180,3,130,2563,0.072222,0.016667,4411446,4409042
4,TB3305,80,7,73,1,348,348,0.095890,0.013699,4411513,4411513


In [120]:
G_BranchStats_DF.sort_values("Number of Recombination Blocks", ascending=False).head(10)

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame
41,Node_20,935,96,839,8,1973,1973,0.114422,0.009535,4411532,4411532
24,Node_3,441,113,328,7,1879,4339,0.344512,0.021341,4411532,4409551
15,TB6599,272,92,180,6,1547,3621,0.511111,0.033333,4411468,4409064
16,TB6976,237,57,180,5,1523,4099,0.316667,0.027778,4411511,4409039
12,TB6786,161,40,121,5,1856,3382,0.330579,0.041322,4411519,4409154
21,TB6846,181,58,123,4,1612,3917,0.471545,0.032520,4411522,4409064
17,TB3572,236,66,170,4,1279,3359,0.388235,0.023529,4411497,4409025
25,Node_4,278,103,175,4,1588,4745,0.588571,0.022857,4411515,4408875
28,Node_7,259,26,233,4,1588,2628,0.111588,0.017167,4411532,4409551
31,Node_10,170,20,150,4,1588,2316,0.133333,0.026667,4411532,4409551


## Annotated Branch stats by lineage

In [121]:
G_BranchStats_DF["Lineage"] = G_BranchStats_DF["Node"].map(node_To_PrimaryLin_Dict).fillna("None")

G_BranchStats_DF["BranchLen"] = G_BranchStats_DF["Node"].map(Tree_BranchLen_Dict).fillna("None")

G_BranchStats_DF["Num_Tips_Downstream"] = G_BranchStats_DF["Node"].map(NumDescendants_PerLeaf_Dict).fillna("None")

G_BranchStats_DF["Total_SNPs"] = G_BranchStats_DF["Total SNPs"]


### Remove last branch for unrooted tree

In [122]:
# Remove the lasts root node, it has no actual branch length

G_BranchStats_Filt_DF = G_BranchStats_DF.query("BranchLen != 'None'")


In [123]:
G_BranchStats_DF.tail(2)

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
41,Node_20,935,96,839,8,1973,1973,0.114422,0.009535,4411532,4411532,lineage4,282.448578,20.0,935
42,Node_21,0,0,0,0,0,0,0.000000,0.000000,4411532,4411532,None,None,None,0


In [124]:
G_BranchStats_Filt_DF.tail(2)

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
40,Node_19,0,0,0,0,0,1973,0.000000,0.000000,4411532,4409551,lineage4,0.004617,17.0,0
41,Node_20,935,96,839,8,1973,1973,0.114422,0.009535,4411532,4411532,lineage4,282.448578,20.0,935


In [125]:
G_BranchStats_Filt_DF.shape[0]

42

In [126]:
G_BranchStats_DF.shape[0]

43

In [127]:
G_BranchStats_DF.query("Total_SNPs == 0")

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
22,Node_1,0,0,0,0,0,0,0.0,0.0,4411513,4411513,lineage2,726.443909,2.0,0
23,Node_2,0,0,0,0,0,4339,0.0,0.0,4411532,4407180,lineage4,0.004617,2.0,0
40,Node_19,0,0,0,0,0,1973,0.0,0.0,4411532,4409551,lineage4,0.004617,17.0,0
42,Node_21,0,0,0,0,0,0,0.0,0.0,4411532,4411532,None,None,None,0


In [128]:
G_BranchStats_DF.query("BranchLen == 0")

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs


In [129]:
G_BranchStats_DF.query("BranchLen == 'None'")

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
42,Node_21,0,0,0,0,0,0,0.0,0.0,4411532,4411532,None,None,None,0


### Output updated branch stats TSV

In [130]:
G_BranchStats_WithLineage_CSV = f"{Gubbins_V1_NEW_ResultsDir}/{Gubbins_OutPrefix}.per_branch_statistics.WithLineagePerNode.tsv"

G_BranchStats_Filt_DF.to_csv(G_BranchStats_WithLineage_CSV, sep="\t", index=False)

#G_BranchStats_WiAnno_CSV = f"{Gubbins_V1_OutputDir}/{Gubbins_OutPrefix}.per_branch_statistics.WiAnno.tsv"
#G_BranchStats_Filt_DF.to_csv(G_BranchStats_WiAnno_CSV, sep="\t", index=False)

In [131]:
!ls -lah $G_BranchStats_WithLineage_CSV

-rw-r--r-- 1 mm774 hpc_farhat 3.8K Sep  1 16:28 /n/data1/hms/dbmi/farhat/mm774/Projects/Mtb-GeneConv/250801.TBP22_22CI.V1/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh/Gubbins.per_branch_statistics.WithLineagePerNode.tsv


In [132]:
G_BranchStats_Filt_DF.head(4)

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
0,TB6733,164,11,153,1,333,2923,0.071895,0.006536,4411490,4409032,lineage4,160.041107,0.0,164
1,TB3898,48,5,43,1,333,2612,0.116279,0.023256,4411532,4409167,lineage4,43.494949,0.0,48
2,TB7340,34,29,5,1,351,3069,5.800000,0.200000,4411532,4408587,lineage4,4.419405,0.0,34
3,TB6977,193,13,180,3,130,2563,0.072222,0.016667,4411446,4409042,lineage4,194.495483,0.0,193


In [133]:
G_BranchStats_Filt_DF.tail(4)

,Node,Total SNPs,Number of SNPs Inside Recombinations,Number of SNPs Outside Recombinations,Number of Recombination Blocks,Bases in Recombinations,Cumulative Bases in Recombinations,r/m,rho/theta,Genome Length,Bases in Clonal Frame,Lineage,BranchLen,Num_Tips_Downstream,Total_SNPs
38,Node_17,206,15,191,2,396,2229,0.078534,0.010471,4411532,4409551,lineage4,210.466324,8.0,206
39,Node_18,76,0,76,0,0,1973,0.000000,0.000000,4411532,4409551,lineage4,75.067871,12.0,76
40,Node_19,0,0,0,0,0,1973,0.000000,0.000000,4411532,4409551,lineage4,0.004617,17.0,0
41,Node_20,935,96,839,8,1973,1973,0.114422,0.009535,4411532,4411532,lineage4,282.448578,20.0,935


# Extras